# Setup

In [1]:
# Setup
import json
import os
import sys
from datetime import datetime, timedelta
import time
from dotenv import load_dotenv, set_key
import random
from pathlib import Path
import logging

# hashing for signing
import hashlib
import hmac

# requests
import requests

# data munging
import pandas as pd
import numpy as np

# helper functions
from tiktok_api_helpers import *

## API Setup

In [2]:
new_access_token = False
use_refresh_token = False

In [3]:
if (new_access_token | use_refresh_token):

    # get new acces_token
    token_url = "https://auth.tiktok-shops.com/api/v2/token/get"

    if use_refresh_token:
        auth_code = os.environ.get("TIKTOK_REFRESH_TOKEN")
        
    params = {
        "app_key": app_key,
        "app_secret": app_secret,
        "auth_code": auth_code,
        "grant_type": "authorized_code",
    }
    
    response = requests.get(token_url, params=params, timeout=15)
    data = response.json()['data']
    access_token = data['access_token']
    print(data)

    # save tokens in .env file
    set_key(".env", "TIKTOK_ACCESS_TOKEN", data['access_token'])
    set_key(".env", "TIKTOK_REFRESH_TOKEN", data['refresh_token'])

else:
    access_token = os.environ.get("TIKTOK_ACCESS_TOKEN")
    refresh_token = os.environ.get("TIKTOK_REFRESH_TOKEN")

## Load Creator Data

In [4]:
# load all creators
df_creators_list = pd.read_excel("creators/all_creators_sorted.xlsx", sheet_name="LIST_CREATOR", usecols=[1, 2, 25])
df_creators = pd.read_csv(CONSOLIDATED_CSV)

In [5]:
df_creators_all = df_creators_list \
    .merge(df_creators, how='left', on="username")

# Extract Top Creators by GMV and Units Sold

In [6]:
all_creators = []
search_key = ""
page_token = ""
page_num = 1

while True:
    result = search_creators_with_retry(
        gmv_ranges=["GMV_RANGE_10000_AND_ABOVE"],
        units_sold_ranges=["UNITS_SOLD_RANGE_100_1000", "UNITS_SOLD_RANGE_1000_AND_ABOVE"],
        search_key=search_key,
        page_token=page_token,
        max_retries=20,
        max_delay=100.0,
    )

    if result.get("code") != 0:
        print(f"  ⚠️  Page {page_num} failed, stopping here: {result}")
        break

    data = result.get("data", {}) or {}
    creators = data.get("creators", [])
    all_creators.extend(creators)

    print(f"Page {page_num}: {len(creators)} creator(s) (total so far: {len(all_creators)})")

    search_key = data.get("search_key", search_key)  # carry forward, per the doc's caching note
    page_token = data.get("next_page_token", "")
    if not page_token:
        break

    page_num += 1
    time.sleep(DELAY_BETWEEN_QUERIES)

print(f"\nDone. {len(all_creators)} total creator(s) collected.")

Page 1: 20 creator(s) (total so far: 20)
Page 2: 20 creator(s) (total so far: 40)
Page 3: 20 creator(s) (total so far: 60)
Page 4: 20 creator(s) (total so far: 80)
Page 5: 20 creator(s) (total so far: 100)
Page 6: 20 creator(s) (total so far: 120)
Page 7: 20 creator(s) (total so far: 140)
Page 8: 20 creator(s) (total so far: 160)
Page 9: 20 creator(s) (total so far: 180)
Page 10: 20 creator(s) (total so far: 200)
Page 11: 20 creator(s) (total so far: 220)
Page 12: 20 creator(s) (total so far: 240)
Page 13: 20 creator(s) (total so far: 260)
Page 14: 20 creator(s) (total so far: 280)
Page 15: 20 creator(s) (total so far: 300)
Page 16: 20 creator(s) (total so far: 320)
Page 17: 20 creator(s) (total so far: 340)
Page 18: 20 creator(s) (total so far: 360)
Page 19: 20 creator(s) (total so far: 380)
Page 20: 20 creator(s) (total so far: 400)
Page 21: 20 creator(s) (total so far: 420)
Page 22: 20 creator(s) (total so far: 440)
Page 23: 20 creator(s) (total so far: 460)
Page 24: 20 creator(s) (

KeyboardInterrupt: 

In [7]:
1+1

2

In [8]:
OUTPUT_DIR = Path("creators")
OUTPUT_DIR.mkdir(exist_ok=True)

RESULTS_CSV = OUTPUT_DIR / "creators_gmv_units_sold.csv"
CHECKPOINT_FILE = OUTPUT_DIR / "creators_gmv_units_sold_checkpoint.json"
LOG_FILE = OUTPUT_DIR / "search_creators_by_gmv_units_sold.log"

In [9]:
logger = logging.getLogger("search_creators_by_gmv_units_sold")
logger.setLevel(logging.INFO)
if not logger.handlers:  # avoid duplicate handlers if this script/module is re-run in the same session
    _file_handler = logging.FileHandler(LOG_FILE, mode="a", encoding="utf-8")
    _file_handler.setFormatter(logging.Formatter("%(asctime)s %(message)s"))
    logger.addHandler(_file_handler)


def log_print(msg: str) -> None:
    """Prints to console AND writes to the log file, so nothing shown live is lost."""
    print(msg)
    logger.info(msg)

In [16]:
# Resume from a previous run if a checkpoint exists, instead of starting
# over from page 1.
if CHECKPOINT_FILE.exists():
    checkpoint = json.loads(CHECKPOINT_FILE.read_text())
    search_key = checkpoint.get("search_key", "")
    page_token = checkpoint.get("page_token", "")
    page_num = checkpoint.get("page_num", 1)
    log_print(f"Resuming from checkpoint: page {page_num}, page_token={page_token!r}")
else:
    search_key = ""
    page_token = ""
    page_num = 1

Resuming from checkpoint: page 37, page_token='b2Zmc2V0PTcwMA=='


In [19]:
df_page = pd.DataFrame(all_creators)

In [ ]:
        file_exists = RESULTS_CSV.exists()
        df_page.to_csv(RESULTS_CSV, mode="a", header=not file_exists, index=False)

In [ ]:
    if creators:
        df_page = pd.DataFrame(creators)
        file_exists = RESULTS_CSV.exists()
        df_page.to_csv(RESULTS_CSV, mode="a", header=not file_exists, index=False)

In [ ]:






all_creators = []

while True:
    result = search_creators_with_retry(
        gmv_ranges=["GMV_RANGE_10000_AND_ABOVE"],
        units_sold_ranges=["UNITS_SOLD_RANGE_100_1000", "UNITS_SOLD_RANGE_1000_AND_ABOVE"],
        search_key=search_key,
        page_token=page_token,
        max_retries=10,
    )

    if result.get("code") != 0:
        log_print(f"  ⚠️  Page {page_num} failed after retries, stopping here. page_token={page_token!r}. Result: {result}")
        break

    data = result.get("data", {}) or {}
    creators = data.get("creators", [])
    all_creators.extend(creators)

    # Save this page's results immediately — so if anything crashes on a
    # LATER page, everything collected so far is already safely on disk.
    if creators:
        df_page = pd.DataFrame(creators)
        file_exists = RESULTS_CSV.exists()
        df_page.to_csv(RESULTS_CSV, mode="a", header=not file_exists, index=False)

    search_key = data.get("search_key", search_key)  # carry forward, per the doc's caching note
    page_token = data.get("next_page_token", "")

    log_print(f"Page {page_num}: {len(creators)} creator(s) (total so far: {len(all_creators)}). page_token={page_token!r}")

    # Save progress AFTER handling this page's data, so the checkpoint
    # always points to the next page still needing to be fetched.
    CHECKPOINT_FILE.write_text(json.dumps({
        "search_key": search_key,
        "page_token": page_token,
        "page_num": page_num + 1,
    }))

    if not page_token:
        break

    page_num += 1
    time.sleep(DELAY_BETWEEN_QUERIES)

log_print(f"\nDone. {len(all_creators)} creator(s) collected this run.")

# Pagination finished cleanly (no more pages, or a page failed and we
# stopped) — clear the checkpoint so a future run starts a fresh search
# instead of "resuming" a search that's actually already done.
if not page_token:
    CHECKPOINT_FILE.unlink(missing_ok=True)
    log_print("Checkpoint cleared — search complete.")
else:
    log_print(f"Checkpoint saved (page_token={page_token!r}) — re-run this script to resume from page {page_num + 1}.")